In [1]:
#1. Librerías.
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import ast
import re
import unidecode

In [ ]:
#2. Constantes.
df_final_path = "./pruebas_batch/df_final_2310_promptcolaborativo_1500samples.csv"
df_exportacion_path = "./pruebas_batch/post_procesamiento_df_final_2310_promptcolaborativo_1500samples.csv"

In [3]:
#3. Lectura.
df_final = pd.read_csv(df_final_path)
#pd.set_option('display.max_columns', None)
#pd.set_option('display.max_rows', None) 

In [84]:
#4. Normalización de columnas de "texto libre".
#a. Formateo.
df_final["sectores_mencionados"] = df_final["sectores_mencionados"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df_final["empresas_mencionadas"] = df_final["empresas_mencionadas"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)

df_final["tickers_mencionados"] = df_final["tickers_mencionados"].apply(
    lambda x: ast.literal_eval(x) if isinstance(x, str) else x
)
#b. Normalizo los sectores.
def normalizar_strings(s):
    s = s.lower().strip() # minúsculas.
    s = unidecode.unidecode(s)  # quita tildes.
    s = s.replace("_", " ").replace("-", " ").replace("/", " ").strip() # reemplazo valores.

    return s

def normalizar_sector_macro(string):
    if not isinstance(string, str):
        return string

    s = normalizar_strings(string)

    # Macro-sectores
    if re.search(r'finan|banca|bono|mercado|fondos|deposito|inversion', s):
        return 'Finanzas'
    if re.search(r'cripto|blockchain|nft|defi|fintech', s):
        return 'Cripto / Fintech'
    if re.search(r'tecnolog|software|data|inteligencia|robot|semiconductor|iot|ia', s):
        return 'Tecnología'
    if re.search(r'energ|petroleo|gas|renovable|gnl|electrico|electricidad', s):
        return 'Energía'
    if re.search(r'min', s) and not 'admin' in s:
        return 'Minería'
    if re.search(r'educa|universidad|escuela|formacion|capacita', s):
        return 'Educación'
    if re.search(r'salud|medic|farmaceut|hospital', s):
        return 'Salud'
    if re.search(r'comerc|retail|supermercado|consumo|alimentos|bebid|frigorif', s):
        return 'Comercio / Consumo'
    if re.search(r'transport|logistic|puerto|aero|hidro|movilidad|vehiculos', s):
        return 'Transporte / Logística'
    if re.search(r'gobier|sector publico|administracion|municipal|provincial', s):
        return 'Gobierno / Sector Público'
    if re.search(r'politi|electoral|legislativ|partid', s):
        return 'Política'
    if re.search(r'justic|judicial|tribunal|juridic', s):
        return 'Justicia'
    if re.search(r'segur', s):
        return 'Seguridad'
    if re.search(r'turism|viaje|hotel|balneario|turistico', s):
        return 'Turismo'
    if re.search(r'industri|manufactur|fabrica|metal|quimic|textil|automotriz', s):
        return 'Industria / Manufactura'
    if re.search(r'agro|ganad|rural|agric|cultivo|frut|soja|maiz|trigo', s):
        return 'Agropecuario'
    if re.search(r'inmob|vivienda|constru|obra', s):
        return 'Construcción / Inmobiliario'
    if re.search(r'medio ambiente|ambient|ecolog|forest', s):
        return 'Medio Ambiente'
    if re.search(r'emple|laboral|trabaj', s):
        return 'Empleo / Laboral'
    if re.search(r'servic', s):
        return 'Servicios'
    if re.search(r'infraestr|hidrovía|hidrovia', s):
        return 'Infraestructura'
    
    # Si no entra en ninguna categoría
    return 'Otros'

df_final["sectores_mencionados"] = df_final["sectores_mencionados"].apply(
    lambda lista: list(dict.fromkeys([normalizar_sector_macro(s) for s in lista])) 
    if isinstance(lista, list) else lista
)

#c. Normalizo nombre de actor principal.
#i. Unifico caracteres.
df_final["nombre_actor_principal"] = df_final["nombre_actor_principal"].apply(
    lambda x: normalizar_strings(x) if isinstance(x, str) else x
)
#ii. Unifico actores que se pueden escribir igual.
mapa_actores = {
    # Javier Milei
    "javier milei": "javier milei",
    "gobierno de javier milei": "javier milei",
    "gobierno nacional (javier milei)": "javier milei",
    "gobierno   javier milei": "javier milei",
    "gobierno (javier milei)": "javier milei",
    "gobierno nacional (presidente javier milei)": "javier milei",
    # Gobierno Nacional general
    "gobierno nacional": "gobierno nacional",
    "gobierno": "gobierno nacional",
    "gobierno argentino": "gobierno nacional",
    # Banco Central
    "banco central": "banco central (bcra)",
    "banco central de la republica argentina": "banco central (bcra)",
    "banco central de la republica argentina (bcra)": "banco central (bcra)",
    "bcra": "banco central (bcra)",
    "banco central (santiago bausili)": "banco central (bcra)",
    # ANSES
    "anses": "anses",
    "administracion nacional de la seguridad social (anses)": "anses",
    "anses (administracion nacional de la seguridad social)": "anses",
    # FMI / Fondo Monetario
    "fondo monetario internacional": "fmi",
    "fondo monetario internacional (fmi)": "fmi",
    "fmi": "fmi",
    # Cristina Kirchner
    "cristina kirchner": "cristina fernandez de kirchner",
    "cristina fernandez de kirchner": "cristina fernandez de kirchner",
    # Luis Caputo
    "luis caputo": "luis caputo",
    "luis \"toto\" caputo": "luis caputo",
    "ministerio de economia   luis caputo": "luis caputo",
    "ministerio de economia (luis caputo)": "luis caputo",
    # La Nación.
    "la nacion": "la nacion",
    "sa la nacion": "la nacion",
    # Otros actores grandes.
    "banco nacion": "banco nacion",
    "banco de la nacion argentina": "banco nacion",
    "banco provincia": "banco provincia",
    "ypf": "ypf",
    "mercado pago": "mercado pago",
    "mercado agroganadero de canuelas": "mercado agroganadero de canuelas",
    "instituto nacional de estadisticas y censos (indec)": "indec",
    "instituto nacional de estadistica y censos (indec)": "indec"
}
#iii. Aplico.
def unificar_actores(actor):
    if not isinstance(actor, str):
        return actor
    actor = mapa_actores.get(actor, actor)  # si está en el mapa, reemplaza
    return actor

df_final["nombre_actor_principal"] = df_final["nombre_actor_principal"].apply(unificar_actores)


#d. Normalizo empresas mencionadas.
#i. Unifico caracteres.
df_final["empresas_mencionadas"] = df_final["empresas_mencionadas"].apply(
    lambda lista: [normalizar_strings(s) for s in lista] if isinstance(lista, list) else lista
)

#ii. Unifico empresas que se pueden escribir de muchas manera.
#1.  Diccionario de unificación.
mapa_empresas = {
    "banco central de la republica argentina": "banco central (bcra)",
    "bcra": "banco central (bcra)",
    "banco central de la republica argentina (bcra)": "banco central (bcra)",
    "banco central": "banco central (bcra)",
    "banco nacion": "banco nacion",
    "banco de la nacion argentina": "banco nacion",
    "la nacion": "la nacion",
    "sa la nacion": "la nacion",
    "lanacion": "la nacion",
    "instituto nacional de estadisticas y censos (indec)": "indec",
    "instituto nacional de estadistica y censos (indec)": "indec"
    # agregr más.
}

#2. Función para unificar nombres dentro de una lista.
def unificar_empresas(lista):
    if isinstance(lista, list):
        return [mapa_empresas.get(s, s) for s in lista]
    return lista

#3. Aplico.
df_final["empresas_mencionadas"] = df_final["empresas_mencionadas"].apply(unificar_empresas)


In [85]:
#5. Analizo los valores únicos por columna.
#i. Sectores mencionales.
todos_los_sectores = [s for lista in df_final["sectores_mencionados"] for s in lista]
sectores_unicos = sorted(set(todos_los_sectores))
print("La cantidad de sectores post-procesamiento es de {}".format(len(sectores_unicos)))
#ii. Empresas mencionadas.
todas_las_empresas = [s for lista in df_final["empresas_mencionadas"] for s in lista]
empresas_unicas = sorted(set(todas_las_empresas))
print("La cantidad de empresas es de {}".format(len(empresas_unicas)))
#iii. Tickers mencionados.
todos_los_tickers = [s for lista in df_final["tickers_mencionados"] for s in lista]
tickers_unicos = sorted(set(todos_los_tickers))
print("La cantidad de tickers es de {}".format(len(tickers_unicos)))
#iv. Tipo de actor principal.
tipo_actor_principal_unicos = df_final["tipo_actor_principal"].unique()
print("La cantidad de tipo de actores principales es de {}".format(len(tipo_actor_principal_unicos)))
#v. Actor principal.
actor_principal = df_final["nombre_actor_principal"].unique()
print("La cantidad de actores principales es de {}".format(len(actor_principal)))
#vi. Caracter.
caracters_unicos = df_final["caracter"].unique()
print("La cantidad de caracteres es de {}".format(len(caracters_unicos)))
#vii. Shock.
shock_unicos = df_final["shock"].unique()
print("La cantidad de shock es de {}".format(len(shock_unicos)))
#viii. Tipo Evento.
tipo_evento_unicos = df_final["tipo_evento"].unique()
print("La cantidad de tipo de evento es de {}".format(len(tipo_evento_unicos)))

La cantidad de sectores post-procesamiento es de 22
La cantidad de empresas es de 1766
La cantidad de tickers es de 114
La cantidad de tipo de actores principales es de 11
La cantidad de actores principales es de 530
La cantidad de caracteres es de 4
La cantidad de shock es de 5
La cantidad de tipo de evento es de 10


In [86]:
#6. Analizo el conteo de valores por columna.
#a. Qué empresas son las de mayor mención?
conteo_empresas = pd.Series(todas_las_empresas).value_counts()
conteo_empresas

banco nacion                                                                                            105
binance                                                                                                  55
banco central (bcra)                                                                                     50
la nacion                                                                                                41
bitso                                                                                                    40
ypf                                                                                                      39
lemon                                                                                                    36
jp morgan                                                                                                23
mercado libre                                                                                            22
indec                       

In [87]:
#b. Cuáles son los actores principales de mayor mención?
df_final["nombre_actor_principal"].value_counts()

nombre_actor_principal
javier milei                                                                                                                                188
gobierno nacional                                                                                                                           147
banco central (bcra)                                                                                                                        122
unknown                                                                                                                                     108
luis caputo                                                                                                                                  36
anses                                                                                                                                        35
donald trump                                                                                                     

In [88]:
#c. Cuáles son los tickers con mayor mención?
conteo_tickers = pd.Series(todos_los_tickers).value_counts()
conteo_tickers

LIBRA          47
BTC            38
AL30           36
ETH            30
SOL            20
LINK           16
ADA            14
AVAX           13
NEAR           12
USDC           12
BNB            12
LTC            12
GD30           11
AL30D          10
USDT           10
AL30C           9
TRUMP           5
XRP             5
DOGE            4
GD35            4
TSLA            3
DAI             3
TRX             3
MELANIA         3
GD46            2
GD41            2
BYMA            2
UNH             1
MANA            1
BUR             1
UNI             1
DOT             1
BCH             1
XLM             1
BA.C            1
GOLD            1
PEP             1
LMT             1
NEM             1
QCOM            1
HD              1
AMX             1
META            1
GM              1
EA              1
XAUT            1
LCAI            1
YPFD            1
BONCER          1
JPM             1
BK              1
BLK             1
HBAR            1
MTPLF           1
SUI             1
ENA       

In [89]:
df_final.columns

Index(['diario', 'fecha', 'titulo', 'contenido', 'url', 'seccion',
       'tipo_actor_principal', 'nombre_actor_principal',
       'empresas_mencionadas', 'tickers_mencionados', 'sectores_mencionados',
       'tipo_evento', 'shock', 'caracter', 'horizonte_dias', 'merval',
       'volatilidad_merval', 'fx_usdars', 'spread_usd', 'tasa_bcra',
       'bonos_soberanos', 'spread_bonos', 'actividad_economica',
       'volumen_mercado', 'valencia_general', 'gobernanza',
       'expectativa_macro_corto', 'expectativa_macro_largo',
       'expectativa_fin_corto', 'expectativa_fin_largo', 'menciona_inflacion',
       'menciona_pbi', 'menciona_reservas', 'menciona_embi', 'menciona_deuda',
       'menciona_fmi', 'menciona_salarios_paritarias', 'menciona_tipo_cambio',
       'menciona_confianza_consumidor', 'menciona_sector_bancario',
       'impacto_sector_bancario', 'menciona_sector_energia',
       'impacto_sector_energia', 'menciona_sector_agroexportador',
       'impacto_sector_agroexportador',

In [90]:
#7. Creo nuevas columnas booleanas con True o False según aparición.
top_empresas = list(conteo_empresas.head(10).keys())
top_tickers = list(conteo_tickers.head(10).keys())
top_actor_principal = list(df_final["nombre_actor_principal"].value_counts().head(10).keys())
 
for actor in top_actor_principal:
    df_final[f'actor_principal_{actor}'] = df_final['nombre_actor_principal'].apply(lambda x: actor in x)

for empresa in top_empresas:
    df_final[f'empresas_{empresa}'] = df_final['empresas_mencionadas'].apply(lambda x: empresa in x)

for ticker in top_tickers:
    df_final[f'ticker_{ticker}'] = df_final['tickers_mencionados'].apply(lambda x: ticker in x)

for sector in sectores_unicos:
    df_final[f'sector_{sector}'] = df_final['sectores_mencionados'].apply(lambda x: sector in x)

for actor in tipo_actor_principal_unicos:
    df_final[f'tipo_actor_principal{actor}'] = df_final['tipo_actor_principal'].apply(lambda x: actor in x)

for caracter in caracters_unicos:
    df_final[f'caracter_{caracter}'] = df_final['caracter'].apply(lambda x: caracter in x)

for shock in shock_unicos:
    df_final[f'shock_{shock}'] = df_final['shock'].apply(lambda x: shock in x)

for tipo_evento in tipo_evento_unicos:
    df_final[f'tipo_evento_{tipo_evento}'] = df_final['tipo_evento'].apply(lambda x: tipo_evento in x)


In [91]:
#8. Nuevas columnas con extensión de la lista.
df_final['cant_empresas'] = df_final['empresas_mencionadas'].apply(len)
df_final['cant_tickers'] = df_final['tickers_mencionados'].apply(len)
df_final['cant_sectores'] = df_final['sectores_mencionados'].apply(len)

In [ ]:
#9. Nuevas columnas con la extensión de la noticia.
df_final['palabras_titulo'] = df_final['titulo'].apply(lambda x: len(str(x).split()))
df_final['palabras_noticia'] = df_final['contenido'].apply(lambda x: len(str(x).split()))

In [93]:
#10. Creo columna booleana con el nombre del diario y la sección.
diarios_unicos = df_final["diario"].unique()
secciones_unicas = df_final["seccion"].unique()

for diario in diarios_unicos:
    df_final[f'diario_{diario}'] = df_final['diario'].apply(lambda x: diario in x)

#for seccion in secciones_unicas:
#    df_final[f'seccion_{seccion}'] = df_final['seccion'].apply(lambda x: seccion in x)

In [94]:
#11. Elimino las columnas que ya no voy a tener en cuenta.
#a. Listado de columnas a eliminar.
columnas_a_eliminar = [
    'diario', 
    'titulo', 
    'contenido', 
    'url', 
    'seccion', 
    'tipo_actor_principal', 
    'nombre_actor_principal', 
    'empresas_mencionadas',
    'tickers_mencionados',
    'sectores_mencionados',
    'tipo_evento',
    'shock',
    'caracter'
]

#b. Eliminación.
df_final = df_final.drop(columns=columnas_a_eliminar)


In [95]:
#12. Exporto.
df_final = df_final.sort_values(by="fecha",ascending=True).reset_index(drop=True)
df_final.to_csv(df_exportacion_path,index=False)